In [ ]:
"""
THE 9 STEPS (executed each stage):

    OUTER LEVEL (Attacker):
      Step 1: Select an attack strategy
      Step 2: Identify reward of this attack (requires inner level)

    INNER LEVEL (Defender):
      Step 3: Select a defend strategy
      Step 4: Calculate optimal IM allocation
      Step 5: Calculate reward for defender (expected saved assets)
      Step 6: Update inner level Q-value

    BACK TO OUTER LEVEL:
      Step 7: Use inner level result as outer level reward
      Step 8: Observe next state (stochastic transition)
      Step 9: Update outer level Q-value using Bellman equation
"""

import numpy as np
import random
from typing import Tuple, List, Dict
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpStatus

# ============================================================================
# PROBLEM PARAMETERS  (
# ============================================================================
TAM        = 20   # Total Attacking Missiles  (TAN=20 from notes)
TIM        = 15   # Total Intercepting Missiles (TIM=15 from notes; TIM/4 = 3.75 per stage)
num_stages = 4    # Number of sequential salvos (T=4 from Colab)
TAM_per_stage = TAM / num_stages   # 5 AMs per stage
TIM_per_stage = TIM / num_stages   # ~3.75 IMs per stage

# Network structure
num_nodes     = 5
all_nodes     = list(range(1, num_nodes + 1))  # [1, 2, 3, 4, 5]
SAM_positions = [1, 4]  # SAM batteries placed at nodes 1 and 4 (D=2 batteries)

coverage_matrix = {
    1: {1: 1, 2: 1, 3: 1, 4: 0, 5: 0},  # Node 1 covers {1,2,3}
    2: {1: 1, 2: 1, 3: 1, 4: 1, 5: 0},  # Node 2 covers {1,2,3,4}
    3: {1: 1, 2: 1, 3: 1, 4: 1, 5: 0},  # Node 3 covers {1,2,3,4}
    4: {1: 0, 2: 1, 3: 1, 4: 1, 5: 1},  # Node 4 covers {2,3,4,5}
    5: {1: 0, 2: 0, 3: 0, 4: 1, 5: 1}   # Node 5 covers {4,5}
}

# Asset parameters
# The notes identify the 2 important assets as nodes n₃ and n₄ in the 5-node network.
# In code they are indexed 0 and 1 (i.e., asset 0 = node 3, asset 1 = node 4).
num_assets    = 2   # 2 important assets (n₃ and n₄ from notes)
asset_indices = list(range(num_assets))  # [0, 1]
asset_value   = 8   # Value of each intact asset

# Probability parameters (from Colab: P=0.85, P_tilde=0.80)
P_AM_HIT  = 0.85   # p  — probability AM destroys asset if not intercepted
P_IM_KILL = 0.80   # p̃  — probability IM destroys an AM

# Initial state
initial_asset_status = [1, 1]   # Both assets intact at start
initial_IM_inventory = {
    1: 8,   # alpha=8 IMs loaded at SAM node 1
    2: 0,
    3: 0,
    4: 8,   # alpha=8 IMs loaded at SAM node 4
    5: 0
}
initial_state = {
    'asset_status': initial_asset_status,
    'IM_inventory': initial_IM_inventory
}

# Learning parameters — defined once here, passed into functions as arguments
# (Q-tables are created inside train_bilevel_qlearning, not here)
gamma         = 0.9    # Discount factor
epsilon_outer = 0.1    # Exploration rate for attacker
epsilon_inner = 0.1    # Exploration rate for defender

# ============================================================================
# MAIN TRAINING LOOP - ONE EPISODE
# ============================================================================

def run_one_episode(Q_outer, Q_inner, N_outer, N_inner, gamma, epsilon_outer, epsilon_inner, verbose: bool = True):
    """
    Execute one complete episode through all stages
    Each stage goes through all 9 steps
    """

    # Initialize episode
    current_state = {
        'asset_status': initial_asset_status.copy(),
        'IM_inventory': dict(initial_IM_inventory)
    }

    TAM_remaining = TAM
    TIM_remaining = TIM
    episode_damage = 0.0

    for stage in range(1, num_stages + 1):

        # Calculate resources for this stage
        stages_left      = num_stages - stage + 1
        TAM_this_stage   = TAM_remaining / stages_left
        TIM_this_stage   = TIM_remaining / stages_left

        if verbose:
            print(f"\n{'='*60}")
            print(f"STAGE {stage}: TAM={TAM_this_stage}, TIM={TIM_this_stage}")
            print(f"Current State: {current_state}")
            print(f"{'='*60}\n")

        state_key = state_to_key(current_state)

        # ====================================================================
        # OUTER LEVEL (ATTACKER) - PART 1
        # ====================================================================

        # --------------------------------------------------------------------
        # STEP 1: SELECT AN ATTACK STRATEGY
        # --------------------------------------------------------------------
        if verbose: print("STEP 1: Select an attack strategy (outer level)")

        feasible_attacks = generate_attack_strategies(current_state['asset_status'], TAM_this_stage)

        for a in feasible_attacks:
            if (state_key, a) not in Q_outer:
                Q_outer[(state_key, a)] = 0.0
            if (state_key, a) not in N_outer:
                N_outer[(state_key, a)] = 0

        attack_action = epsilon_greedy(Q_outer, state_key, feasible_attacks, epsilon_outer)
        N_outer[(state_key, attack_action)] += 1

        if verbose:
            print(f"   Selected attack: {attack_action}")
            print(f"   (This determines how {TAM_this_stage} AMs are distributed across assets)")


        # --------------------------------------------------------------------
        # STEP 2: IDENTIFY THE REWARD OF THIS ATTACK
        # --------------------------------------------------------------------


        # ====================================================================
        # INNER LEVEL (DEFENDER)
        # ====================================================================

        # Construct inner level state (includes attack information)
        inner_state_key = (state_key, attack_action)

        # --------------------------------------------------------------------
        # STEP 3: SELECT A DEFEND STRATEGY
        # --------------------------------------------------------------------

        # Generate all feasible defend actions given attack_action
        feasible_defends = generate_defend_strategies(current_state, TIM_this_stage, attack_action)

        # Select defend action using epsilon-greedy on Q_inner
        for d in feasible_defends:
            if (inner_state_key, d) not in Q_inner:
                Q_inner[(inner_state_key, d)] = 0.0
            if (inner_state_key, d) not in N_inner:
                N_inner[(inner_state_key, d)] = 0

        defend_action = epsilon_greedy(Q_inner, inner_state_key,feasible_defends, epsilon_inner)

        # Increment visit count
        N_inner[(inner_state_key, defend_action)] += 1

        if verbose:
            print(f"   Selected defense: {defend_action}")
            print(f"   (This determines how {TIM_this_stage} IMs are allocated to threatened assets)")


        # --------------------------------------------------------------------
        # STEP 4: CALCULATE OPTIMAL IM ALLOCATION
        # --------------------------------------------------------------------
        if verbose: print("\nSTEP 4: Calculate optimal IM allocation from SAM batteries to assets")

        # Solve optimization model (5-10) to allocate IMs from SAM batteries
        # This determines w_ij (IMs from SAM at i to defend asset j)

        allocation, state_after_alloc, _ = solve_IM_allocation_model(
            current_state,
            attack_action,
            defend_action,
            SAM_positions,
            coverage_matrix,
            verbose=verbose
        )

        if verbose:
            print(f"   IM allocation: {allocation}")
            print(f"   (Maps SAM batteries to threatened assets)")


        # --------------------------------------------------------------------
        # STEP 5: CALCULATE THE REWARD FOR DEFENDER
        # --------------------------------------------------------------------
        if verbose: print("\nSTEP 5: Calculate the reward for defender (expected saved assets)")

        # Calculate expected saved assets using the formula from notes:

        reward_inner = 0
        p_save_per_asset = []
        for i in range(num_assets):
            if current_state['asset_status'][i] == 1:   # only intact assets
                ams = attack_action[i]
                ims = defend_action[i]

                # Calculate saving probability
                p_save = calculate_saving_probability(ams, ims, P_AM_HIT, P_IM_KILL)
                reward_inner += asset_value * p_save
                p_save_per_asset.append(p_save)
            else:
                p_save_per_asset.append(0.0)   # already destroyed

        if verbose: print(f"   Expected saved assets (defender reward): {reward_inner}")


        # --------------------------------------------------------------------
        # STEP 6: UPDATE INNER LEVEL Q-VALUE
        # --------------------------------------------------------------------
        if verbose: print("\nSTEP 6: Update inner level Q-value")

        # Calculate learning rate for inner level
        alpha_inner = 1.0 / N_inner[(inner_state_key, defend_action)]

        # Inner level is single-stage from defender's perspective within
        # this salvo → no future inner Q to look up; target = reward_inner

        Q_inner[(inner_state_key, defend_action)] += alpha_inner * (
            reward_inner - Q_inner[(inner_state_key, defend_action)]
        )

        if verbose:
            print(f"   Updated Q_inner[{inner_state_key}, {defend_action}]")

        # ====================================================================
        # BACK TO OUTER LEVEL (ATTACKER) - PART 2
        # ====================================================================

        # --------------------------------------------------------------------
        # STEP 7: USE INNER LEVEL Q-VALUE AS OUTER LEVEL REWARD
        # --------------------------------------------------------------------
        if verbose:
            print("\n" + "-"*60)
            print("STEP 7: Use inner level result as outer level reward")

        # attacker should be rewarded for damage caused, not assets saved
        reward_outer = (sum(current_state['asset_status']) * asset_value) - reward_inner


        if verbose:
            print(f"\nSTEP 7 — Outer reward (attacker, = damage): {reward_outer:.4f}")


        # --------------------------------------------------------------------
        # STEP 8: OBSERVE NEXT STATE
        # --------------------------------------------------------------------
        if verbose: print("\nSTEP 8: Observe next state (stochastic transition)")

        new_asset_status = list(current_state['asset_status'])
        for i in range(num_assets):
            if current_state['asset_status'][i] == 1:
                ams = attack_action[i]
                ims = defend_action[i]
                p_save = calculate_saving_probability(ams, ims, P_AM_HIT, P_IM_KILL)
                if random.random() >= p_save:        # asset destroyed
                    new_asset_status[i] = 0
                    episode_damage += asset_value

        next_state = {
            'asset_status': new_asset_status,
            'IM_inventory': state_after_alloc['IM_inventory']   # from LP
        }

        if verbose:
            print(f"STEP 8 — Next state: assets={next_state['asset_status']}  "
                  f"IMs={next_state['IM_inventory']}")

        # --------------------------------------------------------------------
        # STEP 9: UPDATE OUTER LEVEL Q-VALUE USING BELLMAN EQUATION
        # --------------------------------------------------------------------
        if verbose: print("\nSTEP 9: Update outer level Q-value using Bellman equation")

        # Calculate learning rate for outer level
        alpha_outer = 1.0 / N_outer[(state_key, attack_action)]

        # Find maximum Q-value for next state (Bellman look-ahead)
        if stage < num_stages:
            next_state_key = state_to_key(next_state)
            next_stages_left = stages_left - 1
            next_TAM_per_stage = (TAM_remaining - TAM_this_stage) / max(next_stages_left, 1)
            next_feasible_attacks = generate_attack_strategies(next_state['asset_status'], next_TAM_per_stage)
            max_Q_next_outer = max(
                Q_outer.get((next_state_key, a), 0.0) for a in next_feasible_attacks
            )
        else:
            max_Q_next_outer = 0.0   # terminal stage

        # Update Q_outer using Bellman equation
        Q_outer[(state_key, attack_action)] += alpha_outer * (
            reward_outer
            + gamma * max_Q_next_outer
            - Q_outer[(state_key, attack_action)]
        )

        if verbose:
            print(f"STEP 9 — Q_outer updated → "
                  f"{Q_outer[(state_key, attack_action)]:.4f}")


        # ====================================================================
        # PREPARE FOR NEXT STAGE
        # ====================================================================
        if verbose:
            print(f"\n{'='*60}")
            print(f"END OF STAGE {stage}")
            print(f"{'='*60}\n")

        # Update state for next stage
        current_state = next_state

        # Update remaining resources (from notes: "TAM = TAM - [TIM/stage]")
        TAM_remaining -= TAM_this_stage
        TIM_remaining -= TIM_this_stage

    # End of episode
    if verbose:
        print("\nEPISODE COMPLETE")
        print(f"All {num_stages} stages executed")
        print(f"Both Q_outer and Q_inner have been updated throughout")

    return episode_damage


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

PRESET_ATTACK_STRATEGIES = [
    (0, 0),  # No attack
    (2, 0),  # 2 AMs on asset 0, 0 AMs on asset 1
    (5, 0),  # 5 AMs on asset 0, 0 AMs on asset 1
    (0, 2),  # 0 AMs on asset 0, 2 AMs on asset 1
    (2, 2),  # 2 AMs on asset 0, 2 AMs on asset 1
    (5, 2),  # 5 AMs on asset 0, 2 AMs on asset 1
    (0, 5),  # 0 AMs on asset 0, 5 AMs on asset 1
    (2, 5),  # 2 AMs on asset 0, 5 AMs on asset 1
    (5, 5),  # 5 AMs on asset 0, 5 AMs on asset 1
]

PRESET_DEFENSE_STRATEGIES = [
    (0, 0),  # No defense
    (1, 0),  # 1 IM on asset 0, 0 IMs on asset 1
    (3, 0),  # 3 IMs on asset 0, 0 IMs on asset 1
    (0, 1),  # 0 IMs on asset 0, 1 IM on asset 1
    (1, 1),  # 1 IM on asset 0, 1 IM on asset 1
    (3, 1),  # 3 IMs on asset 0, 1 IM on asset 1
    (0, 3),  # 0 IMs on asset 0, 3 IMs on asset 1
    (1, 3),  # 1 IM on asset 0, 3 IMs on asset 1
    (3, 3),  # 3 IMs on asset 0, 3 IMs on asset 1
]

def generate_attack_strategies(state, TAM_this_stage):
    """
    Helper for STEP 1: Generate feasible attack strategies

    Args:
        state: Current state of assets, e.g., [1, 1] where 1=intact, 0=destroyed
        TAM_this_stage: Total Attack Modules available (used as constraint check)

    Returns: List of attack actions
    Each attack action is a distribution of TAM_this_stage across assets
    """
    n_assets = len(state)
    feasible_strategies = []

    for attack_strategy in PRESET_ATTACK_STRATEGIES:
        # Check if strategy is valid given current state
        is_valid = True
        total_AMs = 0

        for i in range(n_assets):
            # Can only attack intact assets (state[i] == 1)
            if attack_strategy[i] > 0 and state[i] == 0:
                is_valid = False
                break
            total_AMs += attack_strategy[i]

        # Check if total AMs used doesn't exceed TAM available
        if is_valid and total_AMs <= TAM_this_stage:
            feasible_strategies.append(attack_strategy)

    return feasible_strategies


def generate_defend_strategies(inner_state, TIM_this_stage, attack_action):
    """
    Helper for STEP 3: Generate feasible defend strategies

    Returns: List of defend actions
    Each defend action is a distribution of TIM_this_stage across threatened assets
    """
    n_assets = len(inner_state)
    feasible_strategies = []

    # Identify which assets are being attacked
    threatened_indices = [i for i in range(n_assets) if attack_action[i] > 0]

    # If no attack, return "no defense"
    if not threatened_indices:
        return [tuple([0] * n_assets)]

    for defense_strategy in PRESET_DEFENSE_STRATEGIES:
        is_valid = True
        total_IMs = 0

        for i in range(n_assets):
            # Check various validity conditions:

            # 1. Can only defend assets that are being attacked
            if defense_strategy[i] > 0 and attack_action[i] == 0:
                is_valid = False
                break

            # 2. Optional: Don't over-defend (defense <= attack)
            # Comment this out if your model allows over-defense
            if defense_strategy[i] > attack_action[i]:
                is_valid = False
                break

            total_IMs += defense_strategy[i]

        # 3. Check if total IMs used doesn't exceed TIM available
        if is_valid and total_IMs <= TIM_this_stage:
            feasible_strategies.append(defense_strategy)

    # If no feasible strategies, at least return "no defense"
    if not feasible_strategies:
        feasible_strategies.append(tuple([0] * n_assets))

    return feasible_strategies


def solve_IM_allocation_model(current_state, attack_action, defend_action,
                               SAM_positions, coverage_matrix,
                               p=0.85, p_tilde=0.8, epsilon=0.01, M=100,
                               verbose=False):
    """
    Helper for STEP 4: Solve optimization model to allocate IMs optimally

    This is the "Inner MDP" optimization problem that determines:
    - w_ij: How many IMs to send from SAM at location i to defend asset j

    Args:
        current_state: Dict with 'asset_status' and 'IM_inventory'
                      e.g., {'asset_status': [1, 1], 'IM_inventory': {1: 8, 2: 0, 3: 0, 4: 8, 5: 0}}
        attack_action: Tuple of AMs allocated to each asset, e.g., (5, 2)
        defend_action: Tuple of IMs allocated to each asset, e.g., (1, 1)
        SAM_positions: List of node indices with SAM batteries, e.g., [1, 4]
        coverage_matrix: Dict beta[sam_node][asset_node] = 1 if can cover, 0 otherwise
        p: Probability of successful interception (default: 0.85)
        p_tilde: Alternative probability parameter β̃ (default: 0.8)
        epsilon: Small value to avoid division by zero (default: 0.01)
        M: Big-M constant for coverage constraints (default: 100)
        verbose: If True, print detailed results (default: False)

    Returns:
        allocation: Dict {(sam_node, asset_node): num_IMs}
                   e.g., {(1, 0): 1, (1, 1): 0, (4, 0): 0, (4, 1): 1}
        next_state: Updated state after IM allocation
        objective_value: The optimal objective function value
    """
    ## SET-UP
    num_assets = len(current_state['asset_status'])
    asset_indices = list(range(num_assets))

    # Create node indices from SAM positions
    all_nodes = list(current_state['IM_inventory'].keys())
    node_indices = all_nodes

    # Get current IM inventory at each SAM location
    x_current = current_state['IM_inventory']  # Dict {node: num_IMs}

    # Asset value: uses global asset_value (8 = intact, 0 = destroyed)
    s = {idx: asset_value if current_state['asset_status'][idx] == 1 else 0
         for idx in asset_indices}

    # Map defend_action and attack_action to dictionaries
    z = {asset_idx: defend_action[asset_idx] for asset_idx in asset_indices}
    y = {asset_idx: attack_action[asset_idx] for asset_idx in asset_indices}

    if all(y[m] == 0 for m in asset_indices):
    # No attack — skip LP entirely, no IMs needed
      return {(i,j): 0 for i in node_indices for j in asset_indices}, current_state, 0.0

    ## OPTIMIZATION MODEL
    problem = LpProblem(name="IM_Allocation", sense=LpMaximize)

    indices = [(i, j) for i in node_indices for j in asset_indices]
    v = LpVariable.dicts("v", indices, lowBound=0, cat='Continuous')
    x = LpVariable.dicts("x", node_indices, lowBound=0, cat='Continuous')

    # Pre-compute survival probability for each asset (these are just numbers, not variables)
    prob_save = {}
    for m in asset_indices:
        if y[m] > 0:
            ratio = z[m] / (y[m] + epsilon)
            prob_save[m] = (1 - p * (1 - p_tilde)**ratio)**y[m]
        else:
            prob_save[m] = 1.0  # no attack = asset survives

    # Objective: maximize expected saved asset value (now fully linear in v)
    problem += lpSum(
        v[(i, j)] * prob_save[j] * s[j]
        for i in node_indices
        for j in asset_indices
        if coverage_matrix.get(i, {}).get(j, 0) == 1 and y[j] > 0
)

    # 1. Coverage constraints: Can only send IMs to assets within coverage
    for i in node_indices:
        for j in asset_indices:
            beta_ij = coverage_matrix.get(i, {}).get(j, 0)
            problem += v[(i, j)] <= M * beta_ij

    # 2. Row sum constraints (supply): Total IMs sent from node i cannot exceed inventory
    for i in node_indices:
        problem += lpSum([v[(i, j)] for j in asset_indices]) <= x_current[i]

    # 3. Column sum constraints (demand): Total IMs received at asset j must meet defense strategy
    for j in asset_indices:
        problem += lpSum([v[(i, j)] for i in node_indices]) >= z[j]

    # 4. Slack variable definitions: Remaining IMs after allocation
    for i in node_indices:
        problem += x[i] == x_current[i] - lpSum([v[(i, j)] for j in asset_indices])

    ## SOLVE
    problem.solve()

    if verbose:
        print(f"\nOptimization Status: {LpStatus[problem.status]}")
        print(f"Objective Value: {problem.objective.value()}")

    ## EXTRACT RESULTS
    allocation = {}
    for i in node_indices:
        for j in asset_indices:
            allocation[(i, j)] = v[(i, j)].varValue if v[(i, j)].varValue else 0

    # Update IM inventory for next state
    next_IM_inventory = {}
    for n in node_indices:
        next_IM_inventory[n] = x[n].varValue if x[n].varValue else 0

    # Next state keeps same asset status (actual damage happens in Step 8)
    next_state = {
        'asset_status': current_state['asset_status'].copy(),
        'IM_inventory': next_IM_inventory
    }

    objective_value = problem.objective.value() if problem.objective.value() else 0

    ## RETURN RESULTS  ← CRITICAL!
    return allocation, next_state, objective_value

def calculate_saving_probability(num_AMs, num_IMs, p, p_tilde, epsilon: float = 1e-6):
    """
    Helper for STEP 5: Calculate probability asset survives

    From notes: Uses formula involving (1-P(Loss|s))^(z_i)
    From paper: P_save ≈ (1 - (1-p_d)^(num_IMs/num_AMs) * p_s)^num_AMs
    """
    if num_AMs == 0:
        return 1.0
    if num_IMs == 0:
        return (1.0 - p) ** num_AMs

    ratio   = num_IMs / (num_AMs + epsilon)
    p_save  = (1.0 - p * (1.0 - p_tilde) ** ratio) ** num_AMs
    return max(0.0, min(1.0, p_save))   # clamp to [0, 1]


def epsilon_greedy(Q_table, state_key, feasible_actions, epsilon):
    """
    Helper for STEP 1 and STEP 3: Epsilon-greedy action selection
    """
    if random.random() < epsilon or not feasible_actions:
        return random.choice(feasible_actions)

    # Find action with highest Q-value (default 0.0 for unseen state-action pairs)
    best_action = feasible_actions[0]
    best_q      = Q_table.get((state_key, best_action), 0.0)
    for action in feasible_actions[1:]:
        q = Q_table.get((state_key, action), 0.0)
        if q > best_q:
            best_q      = q
            best_action = action
    return best_action

def state_to_key(state:dict):
    """
    Convert mutable state dict to a hashable key for Q-tables.
    """
    status = tuple(state['asset_status'])
    inventory = tuple(state['IM_inventory'][n] for n in sorted(state['IM_inventory']))
    return (status, inventory)

# ============================================================================
# COMPLETE TRAINING LOOP
# ============================================================================

def train_bilevel_qlearning(num_episodes, verbose_every: int = 100):
    """
    Train the bi-level Q-learning system.
    Each episode runs through all stages; each stage executes all 9 steps.

    Q-tables and visit counts are initialized here (not globally) so there
    is exactly one set of tables in use at a time.
    """

    print("="*70)
    print("BI-LEVEL Q-LEARNING TRAINING")
    print("="*70)

    # Q-tables and visit counts — initialized here, passed into run_one_episode
    Q_outer = {}   # Q_outer[(state_key, attack_action)]  = expected attacker reward
    Q_inner = {}   # Q_inner[(inner_state_key, defend_action)] = expected defender reward
    N_outer = {}   # Visit counts for outer-level learning rate
    N_inner = {}   # Visit counts for inner-level learning rate

    damage_history = []

    for episode in range(1, num_episodes + 1):

        verbose = (episode == 1 or episode % verbose_every == 0)
        if verbose:
            print(f"\n{'#'*70}")
            print(f"EPISODE {episode} / {num_episodes}")
            print(f"{'#'*70}")

        # Run one complete episode through all stages (executes all 9 steps per stage)
        ep_damage = run_one_episode(
            Q_outer, Q_inner, N_outer, N_inner,
            gamma, epsilon_outer, epsilon_inner,
            verbose=verbose
        )
        damage_history.append(ep_damage)

        # Optional: Track and report convergence metrics
        # TODO: Check if Q-values are converging
        # TODO: Optionally decay epsilon over time
        if verbose:
            avg = sum(damage_history[-50:]) / len(damage_history[-50:])
            print(f"\n  → Episode damage: {ep_damage:.1f}  |  "
                  f"Last-50 avg: {avg:.2f}")

    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)

    return Q_outer, Q_inner, damage_history


# ============================================================================
# GREEDY EVALUATION (epsilon = 0)
# ============================================================================

def evaluate_policy(Q_outer: dict, Q_inner: dict, n_eval: int = 100) -> float:
    """Run the learned policy greedily and report average attacker damage."""
    total_damage = 0.0
    for _ in range(n_eval):
        total_damage += run_one_episode(
            Q_outer, Q_inner, {}, {},
            gamma=gamma,
            epsilon_outer=0.0,   # pure exploitation
            epsilon_inner=0.0,
            verbose=False
        )
    avg = total_damage / n_eval
    print(f"\nGREEDY EVALUATION ({n_eval} runs): avg attacker damage = {avg:.2f}")
    return avg

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    print("""
    ============================================================================
    BI-LEVEL Q-LEARNING FOR MISSILE DEFENSE
    ============================================================================

    THE 9 STEPS (executed each stage):

    OUTER LEVEL (Attacker):
      Step 1: Select an attack strategy
      Step 2: Identify reward of this attack (requires inner level)

    INNER LEVEL (Defender):
      Step 3: Select a defend strategy
      Step 4: Calculate optimal IM allocation (Model 5-10)
      Step 5: Calculate reward for defender (expected saved assets)
      Step 6: Update inner level Q-value

    BACK TO OUTER LEVEL:
      Step 7: Use inner level result as outer level reward
      Step 8: Observe next state (stochastic transition)
      Step 9: Update outer level Q-value using Bellman equation

    ============================================================================
    """)

    random.seed(42)
    np.random.seed(42)

    # Train — all parameters are set at the top of the file
    Q_outer, Q_inner, damage_history = train_bilevel_qlearning(
        num_episodes=300,
        verbose_every=100
    )

    # Evaluate learned policies greedily (epsilon=0)
    evaluate_policy(Q_outer, Q_inner, n_eval=200)

    # Summary of learned Q-tables
    print(f"\nQ_outer entries: {len(Q_outer)}")
    print(f"Q_inner entries: {len(Q_inner)}")

    # Show top attacker strategies (highest Q → most damage)
    top_attacks = sorted(Q_outer.items(), key=lambda x: -x[1])[:5]
    print("\nTop 5 attacker strategies (by Q-value):")
    for (s_key, action), q_val in top_attacks:
        print(f"  state={s_key[0]}  attack={action}  Q={q_val:.4f}")
